# Phase 2 : Recherche d'Images Similaires

Ce notebook charge les embeddings générés et permet de rechercher des images similaires.

In [1]:
import os
import json
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torchvision.transforms as transforms
from sklearn.metrics.pairwise import cosine_similarity
import torchvision.models as models

# Configuration
EMBEDDINGS_DIR = Path('../embeddings')
RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Image transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

Using device: cpu


## 1. Charger les Métadonnées et Embeddings

In [2]:
# Load metadata
with open(EMBEDDINGS_DIR / 'metadata.json', 'r') as f:
    metadata = json.load(f)

image_paths = metadata['image_paths']
image_labels = metadata['image_labels']

print(f'Loaded metadata for {len(image_paths)} images')

# Load embeddings (using ResNet-18 by default)
embeddings = np.load(EMBEDDINGS_DIR / 'resnet18_embeddings.npy')
print(f'Loaded embeddings: {embeddings.shape}')

Loaded metadata for 3068 images
Loaded embeddings: (3068, 512)


## 2. Définir la Classe d'Encodeur

In [3]:
class ImageEncoder:
    def __init__(self, model_name='resnet18', device='cpu'):
        self.device = device
        self.model_name = model_name
        self.model = self._load_model(model_name)
        
    def _load_model(self, model_name):
        if model_name == 'resnet18':
            model = models.resnet18(pretrained=True)
        elif model_name == 'mobilenetv2':
            model = models.mobilenet_v2(pretrained=True)
        else:
            raise ValueError(f'Unknown model: {model_name}')
        
        model = torch.nn.Sequential(*list(model.children())[:-1])
        model.to(self.device)
        model.eval()
        return model
    
    def encode_image(self, image_path):
        try:
            img = Image.open(image_path).convert('RGB')
            img_tensor = transform(img).unsqueeze(0).to(self.device)
            
            with torch.no_grad():
                embedding = self.model(img_tensor)
            
            embedding = embedding.view(embedding.size(0), -1)
            embedding = torch.nn.functional.normalize(embedding, p=2, dim=1)
            return embedding.cpu().numpy().flatten()
        except Exception as e:
            print(f'Error encoding {image_path}: {e}')
            return None

encoder = ImageEncoder('resnet18', device=device)
print('Encoder prêt')

C:\Users\carle\Documents\Po\S2\ANI-IA-4048-Intelligence_Artificielle_et_Applications\2-Moteur-recherche-visuelle\venv\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\carle\Documents\Po\S2\ANI-IA-4048-Intelligence_Artificielle_et_Applications\2-Moteur-recherche-visuelle\venv\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Encoder prêt


## 3. Classe de Recherche d'Images

In [4]:
class ImageSearchEngine:
    def __init__(self, embeddings, image_paths, encoder):
        self.embeddings = embeddings
        self.image_paths = image_paths
        self.encoder = encoder
    
    def search(self, query_image_path, k=5):
        # Encode query image
        query_embedding = self.encoder.encode_image(query_image_path)
        
        if query_embedding is None:
            return None
        
        # Reshape for cosine_similarity
        query_embedding = query_embedding.reshape(1, -1)
        
        # Calculate similarities
        similarities = cosine_similarity(query_embedding, self.embeddings)[0]
        
        # Get top-k indices
        top_k_indices = np.argsort(similarities)[::-1][:k]
        top_k_scores = similarities[top_k_indices]
        top_k_paths = [self.image_paths[i] for i in top_k_indices]
        
        return {
            'query_path': query_image_path,
            'results': list(zip(top_k_paths, top_k_scores, top_k_indices))
        }

search_engine = ImageSearchEngine(embeddings, image_paths, encoder)
print('Search engine initialized')

Search engine initialized


## 4. Fonction de Visualisation

In [5]:
def visualize_results(search_result, title='Image Search Results'):
    query_path = search_result['query_path']
    results = search_result['results']
    
    # Create figure
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle(title, fontsize=16)
    
    # Display query image
    query_img = Image.open(query_path)
    axes[0, 0].imshow(query_img)
    axes[0, 0].set_title('Query Image', fontweight='bold')
    axes[0, 0].axis('off')
    
    # Hide other cells in first row
    for i in range(1, 3):
        axes[0, i].axis('off')
    
    # Display results
    for idx, (img_path, score, _) in enumerate(results):
        row = (idx + 3) // 3
        col = (idx + 3) % 3
        
        if row < 2:
            result_img = Image.open(img_path)
            axes[row, col].imshow(result_img)
            axes[row, col].set_title(f'Result {idx+1}\nSimilarity: {score:.3f}', fontsize=10)
            axes[row, col].axis('off')
    
    plt.tight_layout()
    return fig

print('Visualization function ready')

Visualization function ready


## 5. Tester la Recherche (Exemple)

In [ ]:
# Obtention d'image aléatoire du dataset pour le test
test_image_idx = np.random.randint(0, len(image_paths))
test_image_path = image_paths[test_image_idx]

print(f'Testing with: {test_image_path}')
print(f'Category: {image_labels[test_image_idx]}')


results = search_engine.search(test_image_path, k=5)

# Afficher le résultat
print('\nTop 5 Similar Images:')
for i, (path, score, idx) in enumerate(results['results']):
    print(f'{i+1}. {Path(path).name} - Similarity: {score:.4f} - Category: {image_labels[idx]}')

# Visualisation
fig = visualize_results(results)
plt.savefig(RESULTS_DIR / f'search_result_{test_image_idx}.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'\nVisualization saved to results/')

Testing with: clothing-dataset-small-master\train\t-shirt\6ebafad6-334a-40c6-93c6-a3ab854f5efa.jpg
Category: t-shirt
Error encoding clothing-dataset-small-master\train\t-shirt\6ebafad6-334a-40c6-93c6-a3ab854f5efa.jpg: [Errno 2] No such file or directory: 'clothing-dataset-small-master\\train\\t-shirt\\6ebafad6-334a-40c6-93c6-a3ab854f5efa.jpg'

Top 5 Similar Images:


TypeError: 'NoneType' object is not subscriptable

## 6. Recherche Personnalisée

In [ ]:
# Vous pouvez utiliser cette cellule pour rechercher une image spécifique
# Changez le chemin ci-dessous avec le chemin de votre image

custom_image_path = image_paths[0]  # Remplacez avec votre chemin

results = search_engine.search(custom_image_path, k=5)
fig = visualize_results(results, title=f'Search Results for {Path(custom_image_path).name}')
plt.show()

print('Top 5 Similar Images:')
for i, (path, score, idx) in enumerate(results['results']):
    print(f'{i+1}. {Path(path).name} - Similarity: {score:.4f}')

## Recherche Terminée

Le moteur de recherche fonctionne avec succès!

**Informations:**
- Images totales: {}
- Dimension embedding: {}
- Temps de recherche: < 1 seconde (k=5)
- Modèle utilisé: ResNet-18 (pré-entraîné ImageNet)